# Vitrine 06 (Nível Sênior): Inteligência Preditiva de Malha e Proteção de NPS
**Foco:** Machine Learning (Classificação), Logística Avançada, MLOps e Explainable AI (SHAP).

Neste projeto sênior, implementamos um pipeline completo de **Random Forest** para prever atrasos severos em voos comerciais. Ao identificar voos de alto risco, protegemos proativamente o faturamento e o NPS da companhia.

## 1. Importação do Motor de Machine Learning
Em vez de carregar dados pré-processados, instanciamos a classe de pipeline modular `FlightDelayModel` desenvolvida na camada `src`. Isso reflete padrões de engenharia de software em Data Science.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('src'))

import pandas as pd
import plotly.express as px
import shap
from ml_pipeline import FlightDelayModel

# Inicializa e Treina o Modelo
DATA_PATH = r"c:\Users\luizn\OneDrive\Área de Trabalho\Projetos Ciencia de dados\data\processed\pia_2026_advanced_limpo.csv"

ml_engine = FlightDelayModel(DATA_PATH)
ml_engine.load_and_preprocess()
ml_engine.train()

## 2. Validação de Performance e Business Metrics
Avaliação técnica rigorosa do modelo focado em prever atrasos (Severe Delays).

In [ ]:
auc, f1 = ml_engine.evaluate()

## 3. Explainable AI (SHAP): Abrindo a Caixa Preta
Para gerar confiança nas áreas de negócio, utilizamos SHAP Values para entender *exatamente* quais fatores estão forçando o atraso de um voo.

In [ ]:
shap.initjs()
explainer, shap_values, X_sample = ml_engine.explain_shap()

# SHAP Summary Plot (Classe 1 = Atraso Severo)
shap.summary_plot(shap_values[1], X_sample, plot_type="dot")

## 4. Análise de Risco Operacional vs Impacto Financeiro
Aplicando o modelo scorado na base, cruzamos a Probabilidade de Falha com o Revenue em Risco.

In [ ]:
df_scored = ml_engine.score_base()

fig_bubble = px.scatter(df_scored, x="Revenue_USD", y="Severe_Delay_Probability", 
                       size="Passengers", color="Aircraft_Type", 
                       hover_data=["Flight_ID", "Route_Type"],
                       title="<b>Revenue at Risk:</b> Voos Críticos (Alta Receita & Baixa Confiabilidade)",
                       template="plotly_white")
fig_bubble.add_hline(y=0.7, line_dash="dash", line_color="red", annotation_text="Risco Crítico")
fig_bubble.show()

### 🚀 Recomendações Estratégicas (Consultoria SR)
1. **Manutenção Preventiva Direcionada:** Aeronaves com alta probabilidade preditiva (>0.7) devem passar por revisão de sistemas de apoio em solo antes da decolagem.
2. **Proteção de LTV (Life-Time Value):** Passageiros frequentes alocados em voos na zona vermelha de Revenue at Risk devem receber realocação prioritária em caso de quebra de malha.